# Training a DFlash Draft Model for Cosmos3 Nano

This notebook walks through online DFlash training with a Cosmos3 Nano vision-language model as the frozen target. DFlash learns a compact block-diffusion draft model; unlike EAGLE3, it does not use a calibrated draft vocabulary.

| Step | Description |
| :---: | :--- |
| 1 | Install dependencies from this checkout |
| 2 | Optionally authenticate with Hugging Face |
| 3 | Prepare and synthesize the training data |
| 4 | Launch online DFlash training |
| 5 | Export the DFlash checkpoint for deployment |

> **Hardware requirement** – the example launches eight local GPU processes. Set `NUM_GPUS` lower for a smoke test only if the Cosmos3 Nano target fits in the available GPU memory.


## Step 1 – Install Dependencies

Run this notebook from `examples/speculative_decoding/recipes`. The editable install ensures that training uses this checkout's DFlash and VLM support.


In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
python3 -m pip install -r "$REPO_ROOT/examples/speculative_decoding/requirements.txt"
python3 -m pip install -e "$REPO_ROOT[hf]"


## Step 2 – Authenticate with Hugging Face (Optional)

This is needed only when the Cosmos3 Nano checkpoint, processor, or training data is downloaded from the Hub. It is not needed for already-local paths.


In [ ]:
%%bash
hf auth login


## Configure Paths and Dataset Sizes

Set these paths and dataset sizes for your environment before continuing. This cell adds them to the notebook kernel environment, so every subsequent `%%bash` cell inherits them. Individual dataset roots (`PAI_ROOT`, `VQA_ROOT`, and `TEXT_ROOT`) can still be set in the environment to override their default subdirectories under `DATA_ROOT`.


In [ ]:
import os

DATA_ROOT = "/path/to/datasets"
MODEL_PATH = "/path/to/cosmos3-nano-reasoner"
PAI_NUM_GENERATION_SHARDS = 5
VQA_NUM_SAMPLES = 20_000

if DATA_ROOT.startswith("/path/to/") or MODEL_PATH.startswith("/path/to/"):
    raise ValueError("Set DATA_ROOT and MODEL_PATH for your environment before continuing.")

os.environ["DATA_ROOT"] = DATA_ROOT
os.environ["MODEL_PATH"] = MODEL_PATH
os.environ["PAI_NUM_GENERATION_SHARDS"] = str(PAI_NUM_GENERATION_SHARDS)
os.environ["VQA_NUM_SAMPLES"] = str(VQA_NUM_SAMPLES)

## Step 3 – Prepare and Synthesize the Training Data

DFlash training benefits most from target-model completions rather than a large collection of human-written answers. Build the dataset from three complementary sources, then ask the frozen Cosmos3 Nano target to synthesize the assistant turns:

1. **Representative usage samples** – a small PAI-Understanding slice stands in for real production traffic.
2. **Visual reasoning data** – a larger VQA v2 image-question set reinforces image understanding.
3. **High-quality text prompts** – a large multilingual prompt set preserves broad language capability.

The first two sources retain their media; the text source is text-only. The final merge normalizes every row to OpenAI-style `messages`, resolves media to absolute paths, and can remove exact or prompt-level duplicates. This is why Step 4 uses `data.vlm_img_dir=/`.

> **Helper prerequisite.** The commands below use `recipes/prepare_multimodal_synthetic_shards.py`, `recipes/run_multimodal_synthetic_generation.sh`, and the accompanying `distributed_generate/` helpers. Keep that helper set in this checkout before running the cells.

For DFlash, choose a `training_seq_len` divisible by `dflash_block_size` (this example uses `16384` and `8`). Keep the model's ChatML template or provide a compatible assistant-mask template when using `answer_only_loss=true`.


### 3.1 Representative Usage: PAI-Understanding

Use a modest representative sample. In a production setting, replace this benchmark with a privacy-reviewed sample of real requests when available. The benchmark's reference answers are retained only as metadata; the target model produces the assistant response used for training.


In [ ]:
%%bash
# Login node: download PAI-Understanding and make native-video prompt shards.
set -euo pipefail
SPEC="$(cd .. && pwd)"
: "${DATA_ROOT:?Run the Configure Paths cell before this cell.}"
PAI_ROOT=${PAI_ROOT:-$DATA_ROOT/pai_understanding}
PAI_SHARDS="$SPEC/CR3_data/pai_understanding_native_shards"

hf download shi-labs/physical-ai-bench-understanding \
  --repo-type dataset \
  --local-dir "$PAI_ROOT" \
  --max-workers 8

python3 "$SPEC/recipes/prepare_multimodal_synthetic_shards.py" \
  --dataset pai_understanding \
  --dataset_dir "$PAI_ROOT" \
  --media_root "$PAI_ROOT" \
  --output_dir "$PAI_SHARDS"


In [ ]:
%%bash
# Compute node: run a representative PAI shard slice in an existing Slurm GPU allocation.
set -euo pipefail
SPEC="$(cd .. && pwd)"
: "${DATA_ROOT:?Run the Configure Paths cell before this cell.}"
: "${MODEL_PATH:?Run the Configure Paths cell before this cell.}"
PAI_ROOT=${PAI_ROOT:-$DATA_ROOT/pai_understanding}
export MODEL_PATH
export DATASET_DIR="$PAI_ROOT"
export MEDIA_ROOT="$PAI_ROOT"
export SHARD_PATH="$SPEC/CR3_data/pai_understanding_native_shards"
export PREPARE_SHARDS=0
NODE_NAMES="$(scontrol show hostnames "$SLURM_JOB_NODELIST" | paste -sd, -)"
: "${PAI_NUM_GENERATION_SHARDS:?Run the Configure Paths and Dataset Sizes cell before this cell.}"
PAI_START_SHARD=${PAI_START_SHARD:-0}
PAI_NUM_AVAILABLE_SHARDS=$(find "$SHARD_PATH" -maxdepth 1 -type f -name 'train-*.jsonl' | wc -l)
PAI_NUM_NODES=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | wc -l)
[ "$PAI_NUM_AVAILABLE_SHARDS" -gt 0 ] || { echo "No PAI shards found in $SHARD_PATH" >&2; exit 1; }
[ "$PAI_NUM_NODES" -gt 0 ] || { echo "No allocated nodes found" >&2; exit 1; }
(( PAI_START_SHARD + PAI_NUM_GENERATION_SHARDS <= PAI_NUM_AVAILABLE_SHARDS )) || { echo "Requested PAI shard range exceeds $PAI_NUM_AVAILABLE_SHARDS available shards" >&2; exit 1; }
(( PAI_NUM_GENERATION_SHARDS % PAI_NUM_NODES == 0 )) || { echo "$PAI_NUM_GENERATION_SHARDS selected PAI shards cannot be distributed evenly over $PAI_NUM_NODES nodes" >&2; exit 1; }
PAI_SHARDS_PER_NODE=$(( PAI_NUM_GENERATION_SHARDS / PAI_NUM_NODES ))

"$SPEC/recipes/run_multimodal_synthetic_generation.sh" \
  pai_understanding "$SLURM_JOB_ID" "$PAI_START_SHARD" "$PAI_SHARDS_PER_NODE" "$NODE_NAMES"


### 3.2 Visual Reasoning: VQA v2

Make this component larger than the representative sample. The example below builds 20,000 train-split prompts; increase `--num_samples` when the target workload needs more visual coverage.


In [ ]:
%%bash
# Login node: fetch the VQA v2 train questions, annotations, and COCO images.
set -euo pipefail
SPEC="$(cd .. && pwd)"
: "${DATA_ROOT:?Run the Configure Paths cell before this cell.}"
VQA_ROOT=${VQA_ROOT:-$DATA_ROOT/vqa_v2}
IMAGE_ROOT="$VQA_ROOT/images"
: "${VQA_NUM_SAMPLES:?Run the Configure Paths and Dataset Sizes cell before this cell.}"
VQA_SHARD_PATH=${VQA_SHARD_PATH:-$SPEC/CR3_data/vqa_v2_train_shards}

mkdir -p "$VQA_ROOT" "$IMAGE_ROOT"
curl -L --fail --retry 5 -C - -o "$VQA_ROOT/v2_Questions_Train_mscoco.zip" \
  https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip
curl -L --fail --retry 5 -C - -o "$VQA_ROOT/v2_Annotations_Train_mscoco.zip" \
  https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip
curl -L --fail --retry 5 -C - -o "$IMAGE_ROOT/train2014.zip" \
  http://images.cocodataset.org/zips/train2014.zip
unzip -n "$VQA_ROOT/v2_Questions_Train_mscoco.zip" -d "$VQA_ROOT"
unzip -n "$VQA_ROOT/v2_Annotations_Train_mscoco.zip" -d "$VQA_ROOT"
unzip -n "$IMAGE_ROOT/train2014.zip" -d "$IMAGE_ROOT"

python3 "$SPEC/recipes/prepare_multimodal_synthetic_shards.py" \
  --dataset vqa_v2 \
  --vqa_root "$VQA_ROOT" \
  --image_root "$IMAGE_ROOT" \
  --vqa_splits train \
  --num_samples "$VQA_NUM_SAMPLES" \
  --output_dir "$VQA_SHARD_PATH"


In [ ]:
%%bash
# Compute node: distribute all prepared VQA shards across the allocated nodes.
set -euo pipefail
SPEC="$(cd .. && pwd)"
: "${DATA_ROOT:?Run the Configure Paths cell before this cell.}"
: "${MODEL_PATH:?Run the Configure Paths cell before this cell.}"
export MODEL_PATH
VQA_ROOT=${VQA_ROOT:-$DATA_ROOT/vqa_v2}
IMAGE_ROOT="$VQA_ROOT/images"
export VQA_ROOT IMAGE_ROOT
export SHARD_PATH=${VQA_SHARD_PATH:-$SPEC/CR3_data/vqa_v2_train_shards}
export PREPARE_SHARDS=0
NODE_NAMES="$(scontrol show hostnames "$SLURM_JOB_NODELIST" | paste -sd, -)"
VQA_NUM_SHARDS=$(find "$SHARD_PATH" -maxdepth 1 -type f -name '*.jsonl' | wc -l)
VQA_NUM_NODES=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | wc -l)
VQA_START_SHARD=${VQA_START_SHARD:-0}
[ "$VQA_NUM_SHARDS" -gt 0 ] || { echo "No VQA shards found in $SHARD_PATH" >&2; exit 1; }
[ "$VQA_NUM_NODES" -gt 0 ] || { echo "No allocated nodes found" >&2; exit 1; }
(( VQA_NUM_SHARDS % VQA_NUM_NODES == 0 )) || { echo "$VQA_NUM_SHARDS shards cannot be distributed evenly over $VQA_NUM_NODES nodes" >&2; exit 1; }
VQA_SHARDS_PER_NODE=$(( VQA_NUM_SHARDS / VQA_NUM_NODES ))

"$SPEC/recipes/run_multimodal_synthetic_generation.sh" \
  vqa_v2 "$SLURM_JOB_ID" "$VQA_START_SHARD" "$VQA_SHARDS_PER_NODE" "$NODE_NAMES"


### 3.3 High-Quality Text: Synthetic Multilingual Prompts

Use a large text-only component unless the deployment is deliberately narrow. `nvidia/Speculative-Decoding-Multilingual-Prompt-v2` provides prompts; Cosmos3 Nano generates the assistant answers, so this remains synthetic training data. We use eight temperatures to increase response diversity.

A curated subset of `nvidia/Nemotron-Post-Training-Dataset-v2` (chat split) is also useful as an optional non-synthetic baseline. The bundled `nemotron_mapping.bin` holds 0-based little-endian packed `int32` row indices and selects 89,511 conversations. Do not add that direct-human-answer file to the final merge below if the training mix must remain synthetic.


In [ ]:
%%bash
# Optional non-synthetic baseline; run from this recipes directory.
# The mapping file is bundled alongside this notebook.
set -euo pipefail
python3 ../../dataset/add_nemotron_chat.py --mapping-file nemotron_mapping.bin
count=$(wc -l < input_conversations/nemotron-chat.jsonl)
echo "${count} conversations in input_conversations/nemotron-chat.jsonl"
[ "$count" -eq 89511 ] || { echo "ERROR: expected 89511, got $count"; exit 1; }


In [ ]:
%%bash
# Login node: download the text prompts and create full prompt shards.
set -euo pipefail
SPEC="$(cd .. && pwd)"
: "${DATA_ROOT:?Run the Configure Paths cell before this cell.}"
TEXT_ROOT=${TEXT_ROOT:-$DATA_ROOT/specdec_multilingual_prompt}
TEXT_SHARDS="$SPEC/CR3_data/specdec_multilingual_prompt_full_shards"

hf download nvidia/Speculative-Decoding-Multilingual-Prompt-v2 \
  --repo-type dataset \
  --local-dir "$TEXT_ROOT" \
  --max-workers 8

python3 "$SPEC/recipes/prepare_multimodal_synthetic_shards.py" \
  --dataset specdec_multilingual_prompt \
  --text_data "$TEXT_ROOT" \
  --max_lines_per_shard 1024 \
  --overwrite \
  --output_dir "$TEXT_SHARDS"


In [ ]:
%%bash
# Compute node: use the allocation dynamically; no fixed node list is needed.
# START_SHARD=1059 is a resume example. Set it to 0 for a new full run.
set -euo pipefail
SPEC="$(cd .. && pwd)"
: "${DATA_ROOT:?Run the Configure Paths cell before this cell.}"
: "${MODEL_PATH:?Run the Configure Paths cell before this cell.}"
export MODEL_PATH
TEXT_ROOT=${TEXT_ROOT:-$DATA_ROOT/specdec_multilingual_prompt}
export TEXT_DATA="$TEXT_ROOT"
export SHARD_PATH="$SPEC/CR3_data/specdec_multilingual_prompt_full_shards"
export OUTPUT_PATH="$SPEC/specdec_multilingual_prompt_full_v024_outputs"
export PREPARE_SHARDS=0
export BACKEND=vllm
export CONTAINER_IMAGE=vllm/vllm-openai:v0.24.0
export SGLANG_TP_SIZE=1
export NUM_TEMPERATURES=8
NODE_NAMES="$(scontrol show hostnames "$SLURM_JOB_NODELIST" | paste -sd, -)"
START_SHARD=${START_SHARD:-0}
JOBS_PER_NODE=${JOBS_PER_NODE:-1000}

"$SPEC/recipes/run_multimodal_synthetic_generation.sh" \
  specdec_multilingual_prompt "$SLURM_JOB_ID" "$START_SHARD" "$JOBS_PER_NODE" "$NODE_NAMES"


### 3.4 Merge, Resolve Media, and Optionally Deduplicate

Run this after all three synthesis jobs finish. It creates one atomic JSONL with absolute media paths, which permits VQA images and PAI videos to coexist with `data.vlm_img_dir=/`. `DEDUP_MODE=exact` (default) removes only identical conversations, preserving temperature diversity. Set `DEDUP_MODE=prompt` to retain one synthetic completion per input prompt instead.


In [ ]:
%%bash
set -euo pipefail
SPEC="$(cd .. && pwd)"
: "${DATA_ROOT:?Run the Configure Paths cell before this cell.}"
PAI_ROOT=${PAI_ROOT:-$DATA_ROOT/pai_understanding}
VQA_ROOT=${VQA_ROOT:-$DATA_ROOT/vqa_v2}
PAI_OUTPUT=${PAI_OUTPUT:-$SPEC/pai_understanding_synthetic_outputs}
VQA_OUTPUT=${VQA_OUTPUT:-$SPEC/vqa_v2_synthetic_outputs}
TEXT_OUTPUT=${TEXT_OUTPUT:-$SPEC/specdec_multilingual_prompt_full_v024_outputs}
TRAINING_DATA=${TRAINING_DATA:-$SPEC/CR3_data/cosmos3_nano_dflash_train.jsonl}
DEDUP_MODE=${DEDUP_MODE:-exact}  # exact or prompt

python3 - "$PAI_OUTPUT" "$VQA_OUTPUT" "$TEXT_OUTPUT" "$PAI_ROOT" "$VQA_ROOT/images" "$TRAINING_DATA" "$DEDUP_MODE" <<'PY'
from __future__ import annotations

import hashlib
import json
import os
import sys
from collections import Counter
from pathlib import Path

pai_output, vqa_output, text_output, pai_root, vqa_root, output_path, dedup_mode = [*map(Path, sys.argv[1:7]), sys.argv[7]]
if dedup_mode not in {'exact', 'prompt'}:
    raise ValueError('DEDUP_MODE must be exact or prompt')

sources = (
    ('pai_understanding', pai_output, pai_root),
    ('vqa_v2', vqa_output, vqa_root),
    ('specdec_multilingual_prompt', text_output, None),
)

def records(directory: Path):
    if not directory.is_dir():
        raise FileNotFoundError(f'Missing synthetic-output directory: {directory}')
    for path in sorted(directory.rglob('*.jsonl')):
        with path.open(encoding='utf-8') as file:
            for line_number, line in enumerate(file, start=1):
                if line.strip():
                    try:
                        yield path, line_number, json.loads(line)
                    except json.JSONDecodeError as error:
                        raise ValueError(f'Invalid JSON at {path}:{line_number}') from error

def normalize_messages(record: dict, media_root: Path | None):
    messages = record.get('messages') or record.get('conversations')
    if not isinstance(messages, list) or record.get('generation_error'):
        return None
    normalized = []
    for message in messages:
        if not isinstance(message, dict):
            return None
        role = message.get('role')
        content = message.get('content')
        if role not in {'system', 'user', 'assistant'} or not isinstance(content, (str, list)):
            return None
        if isinstance(content, str):
            normalized.append({'role': role, 'content': content})
            continue
        parts = []
        for part in content:
            if not isinstance(part, dict) or part.get('type') not in {'text', 'image', 'video'}:
                return None
            part = dict(part)
            key = part['type']
            if key in {'image', 'video'} and isinstance(part.get(key), str) and media_root is not None:
                part[key] = str((media_root / part[key]).resolve()) if not os.path.isabs(part[key]) else part[key]
            parts.append(part)
        normalized.append({'role': role, 'content': parts})
    if not any(message['role'] == 'user' for message in normalized):
        return None
    if not any(message['role'] == 'assistant' and str(message['content']).strip() for message in normalized):
        return None
    return normalized

output_path.parent.mkdir(parents=True, exist_ok=True)
temporary = output_path.with_suffix(output_path.suffix + '.tmp')
seen: set[str] = set()
written = Counter()
skipped = Counter()
with temporary.open('w', encoding='utf-8') as destination:
    for source_name, directory, media_root in sources:
        for _, _, record in records(directory):
            messages = normalize_messages(record, media_root)
            if messages is None:
                skipped['invalid_or_failed'] += 1
                continue
            identity = [message for message in messages if message['role'] != 'assistant'] if dedup_mode == 'prompt' else messages
            fingerprint = hashlib.sha256(json.dumps(identity, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
            if fingerprint in seen:
                skipped['duplicate'] += 1
                continue
            seen.add(fingerprint)
            json.dump({'id': record.get('id'), 'dataset': record.get('dataset', source_name), 'messages': messages}, destination, ensure_ascii=False)
            destination.write('\n')
            written[source_name] += 1

os.replace(temporary, output_path)
print(f'Wrote {sum(written.values())} conversations to {output_path}')
print(f'By source: {dict(written)}; skipped: {dict(skipped)}')
PY

wc -l "$TRAINING_DATA"


## Step 4 – Train the DFlash Draft Model

DFlash uses the online speculative-decoding recipe directly. It loads Cosmos3 Nano as the target, collects its hidden states through the top-level VLM forward, and trains a five-layer Qwen3-style draft decoder.

The defaults below use the Step 3 merged JSONL and the Cosmos3 Nano checkpoint set in Configure Paths. Override the corresponding environment variables when those paths differ. The architecture settings are pinned to Cosmos3 Nano's text tower rather than inferred from generic Qwen3 defaults.


In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"

SPEC="$REPO_ROOT/examples/speculative_decoding"
: "${MODEL_PATH:?Run the Configure Paths cell before this cell.}"
TRAINING_DATA=${TRAINING_DATA:-$SPEC/CR3_data/cosmos3_nano_dflash_train.jsonl}
VLM_IMG_DIR=/
OUTPUT_DIR="$REPO_ROOT/ckpts/cosmos3-nano-dflash"
NUM_GPUS=${NUM_GPUS:-8}

[ -d "$MODEL_PATH" ] || { echo "Missing MODEL_PATH: $MODEL_PATH" >&2; exit 1; }
[ -s "$TRAINING_DATA" ] || { echo "Missing Step 3 training JSONL: $TRAINING_DATA" >&2; exit 1; }

export WANDB_MODE=disabled
export TOKENIZERS_PARALLELISM=false

cd "$REPO_ROOT"
torchrun --nproc_per_node="$NUM_GPUS" \
  examples/speculative_decoding/main.py \
  --config modelopt_recipes/general/speculative_decoding/dflash.yaml \
  model.model_name_or_path="$MODEL_PATH" \
  model.trust_remote_code=true \
  data.data_path="$TRAINING_DATA" \
  data.vlm_processor="$MODEL_PATH" \
  data.vlm_img_dir="$VLM_IMG_DIR" \
  training.output_dir="$OUTPUT_DIR" \
  training.num_train_epochs=25 \
  training.per_device_train_batch_size=1 \
  training.gradient_accumulation_steps=2 \
  training.training_seq_len=16384 \
  training.answer_only_loss=true \
  training.save_steps=500 \
  training.logging_steps=10 \
  training.dataloader_num_workers=2 \
  training.dataloader_prefetch_factor=2 \
  training.ddp_find_unused_parameters=false \
  training.report_to=none \
  dflash.dflash_block_size=8 \
  dflash.dflash_num_anchors=128 \
  dflash.dflash_loss_objective=decay \
  dflash.dflash_loss_decay_factor=4 \
  dflash.dflash_architecture_config.num_hidden_layers=5 \
  dflash.dflash_architecture_config.num_attention_heads=32 \
  dflash.dflash_architecture_config.num_key_value_heads=8 \
  dflash.dflash_architecture_config.head_dim=128 \
  dflash.dflash_architecture_config.intermediate_size=12288 \
  dflash.dflash_architecture_config.max_position_embeddings=262144 \
  dflash.dflash_architecture_config.rms_norm_eps=1e-06 \
  dflash.dflash_architecture_config.rope_theta=5000000 \
  dflash.dflash_mask_token_id=151669


## Step 5 – Export the DFlash Checkpoint

Export the selected training checkpoint to the DFlash Hugging Face format consumed by vLLM. Replace `checkpoint-<step>` with a saved checkpoint directory.


In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
CKPT_DIR="$REPO_ROOT/ckpts/cosmos3-nano-dflash/checkpoint-<step>"
EXPORT_PATH="$REPO_ROOT/export/cosmos3-nano-dflash"

python3 "$REPO_ROOT/examples/speculative_decoding/scripts/export_hf_checkpoint.py" \
  --model_path "$CKPT_DIR" \
  --export_path "$EXPORT_PATH" \
  --trust_remote_code


## Deployment

Serve the Cosmos3 Nano target with the exported DFlash draft. A DFlash block size of eight yields seven speculative tokens (one position is the context/bonus token).

```bash
vllm serve "$MODEL_PATH" \
  --speculative-config "{\"method\": \"dflash\", \"model\": \"$EXPORT_PATH\", \"num_speculative_tokens\": 7}"
```
